# Multistage & Mission Architecture

Demonstrates the multistage/mission architecture as it is built up commit by commit (see `mission_multistage_design.md`). Each section below corresponds to one landed commit and is added to as the next commit lands.

**Landed so far:**
1. `Stage` — wraps a single-stage `Rocket`
2. `Deployable` — a carried payload released mid-flight
3. `MultiStageRocket` — composes a stage + its deployables into one flight-ready `Rocket` (single active stage only, for now)


In [ ]:
from rocketpy import NoseCone, Rocket, SolidMotor
from rocketpy.rocket.multistage import Deployable, MultiStageRocket, Stage

## Building a base rocket

`Stage` and `MultiStageRocket` wrap ordinary `Rocket` objects — nothing new is needed to build one. This is the same Calisto rocket used throughout the RocketPy docs.

In [ ]:
Pro75M1670 = SolidMotor(
    thrust_source="../../data/motors/cesaroni/Cesaroni_M1670.eng",
    dry_mass=1.815,
    dry_inertia=(0.125, 0.125, 0.002),
    nozzle_radius=33 / 1000,
    grain_number=5,
    grain_density=1815,
    grain_outer_radius=33 / 1000,
    grain_initial_inner_radius=15 / 1000,
    grain_initial_height=120 / 1000,
    grain_separation=5 / 1000,
    grains_center_of_mass_position=0.397,
    center_of_dry_mass_position=0.317,
    nozzle_position=0,
    burn_time=3.9,
    throat_radius=11 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

calisto = Rocket(
    radius=127 / 2000,
    mass=14.426,
    inertia=(6.321, 6.321, 0.034),
    power_off_drag="../../data/rockets/calisto/powerOffDragCurve.csv",
    power_on_drag="../../data/rockets/calisto/powerOnDragCurve.csv",
    center_of_mass_without_motor=0,
    coordinate_system_orientation="tail_to_nose",
)
calisto.add_motor(Pro75M1670, position=-1.255)

print(f"calisto.dry_mass = {calisto.dry_mass:.4f} kg")
print(f"calisto.motor.burn_out_time = {calisto.motor.burn_out_time} s")

## `Stage`: wrapping a rocket as one stage of a vehicle

For now `Stage` is a thin wrapper: `dry_mass` and `burn_out_time` just forward to the wrapped `Rocket`'s own already-computed attributes. It becomes meaningful once `MultiStageRocket` composes several stages together (a later commit).

In [ ]:
booster = Stage(name="booster", rocket=calisto)

print(f"booster.dry_mass = {booster.dry_mass:.4f} kg")
print(f"booster.burn_out_time = {booster.burn_out_time} s")

assert booster.dry_mass == calisto.dry_mass
assert booster.burn_out_time == calisto.motor.burn_out_time

## `Deployable`: a payload carried and ejected mid-flight

A `Deployable` contributes only mass and inertia while attached. Its free-flight aerodynamics come from either a fully built `free_rocket`, or surfaces added one at a time with `add_surface()` — the two are mutually exclusive, and `add_surface()` requires `radius` to be set.

In [ ]:
payload = Deployable(
    name="payload",
    mass=4.5,
    inertia=(0.1, 0.1, 0.001),
    position=1.10,
    radius=0.05,
)

print(f"payload.mass = {payload.mass} kg")
print(f"payload.position = {payload.position} m")
print(f"payload.surfaces (before add_surface) = {payload.surfaces}")

In [ ]:
# add_surface requires radius to be set
no_radius_payload = Deployable(
    name="no_radius_payload", mass=1.0, inertia=(0, 0, 0), position=0.5
)
nose = NoseCone(length=0.2, kind="vonKarman", base_radius=0.05, rocket_radius=0.05)
try:
    no_radius_payload.add_surface(nose, position=0.1)
except ValueError as error:
    print(f"Raised as expected: {error}")

In [ ]:
# add_surface and free_rocket are mutually exclusive
payload.add_surface(nose, position=0.1)
print(f"payload.surfaces (after add_surface) = {payload.surfaces}")

## `MultiStageRocket`: composing a stage + deployable into one flight-ready `Rocket`

`flight_rocket()` composes the wrapped stage's mass/inertia/CoM with every deployable still aboard (parallel axis theorem), attaches the stage's motor, and reuses its aerodynamic surfaces and drag curve. Only a single active stage is supported so far — multi-stage composition (booster + sustainer together) is a later commit.

In [ ]:
vehicle = MultiStageRocket(stages=[booster])
deployable = vehicle.add_deployable(
    name="payload", mass=4.5, inertia=(0.1, 0.1, 0.001), position=1.10
)

flight_rocket = vehicle.flight_rocket(
    active_stages=(booster,), carried_deployables=(deployable,)
)

# Hand-computed weighted average, independent of flight_rocket's own
# code path — same check used in tests/unit/rocket/test_multistage.py
expected_mass = calisto.mass + 4.5
expected_center_of_mass = (
    calisto.mass * calisto.center_of_mass_without_motor + 4.5 * 1.10
) / expected_mass

print(f"flight_rocket.mass = {flight_rocket.mass:.4f} kg (expected {expected_mass:.4f})")
print(
    f"flight_rocket.center_of_mass_without_motor = {flight_rocket.center_of_mass_without_motor:.4f} m "
    f"(expected {expected_center_of_mass:.4f})"
)

assert abs(flight_rocket.mass - expected_mass) < 1e-9
assert abs(
    flight_rocket.center_of_mass_without_motor - expected_center_of_mass
) < 1e-9

## Not built yet

Deliberately out of scope for the commits so far (see the roadmap in `mission_multistage_design.md`):

- `flight_rocket()` for more than one active stage (booster + sustainer composed together, surfaces repositioned into stack coordinates)
- `Mission` — the orchestrator that runs one `Flight` per vehicle configuration with state handoff between them
- Deterministic time-based separation (motor burnout + delay) and apogee-triggered deployable ejection
- `StochasticMission`

Each lands as its own commit with its own tests; this notebook grows alongside them.